# Haploid vs diploid maize seeds: ResNet18 + QCNN training on Kaggle

Before running, in the right-hand panel:
1. **Input → Add Input**: add your uploaded dataset (it contains `code/` and `data/`).
2. **Session options → Accelerator**: **GPU T4** (not P100; recent PyTorch builds may not support it).
3. **Session options → Internet**: **On** (needed to install PennyLane and download the ResNet18 weights).

Then **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the browser.
When it finishes, download `results.zip` from the **Output** tab.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch

assert torch.cuda.is_available(), "No GPU found: set Accelerator to GPU T4 in the side panel."
gpu = torch.cuda.get_device_name(0)
print("PyTorch", torch.__version__, "| GPU:", gpu)

In [ ]:
%pip install -q pennylane

In [ ]:
# Find the uploaded data and code, wherever Kaggle mounted them.
INPUT = Path("/kaggle/input")


def find_dir(test, what):
    hits = sorted(p for p in INPUT.rglob("*") if p.is_dir() and test(p))
    if not hits:
        raise FileNotFoundError(f"could not find {what} under {INPUT}; did you add the dataset as an input?")
    return hits[0]


data_dir = find_dir(lambda p: (p / "haploid").is_dir() and (p / "diploid").is_dir(), "the haploid/diploid folders")
code_dir = find_dir(lambda p: (p / "04_finetune.py").is_file(), "the project code")
counts = {c: len(list((data_dir / c).glob("*.jpg"))) for c in ["haploid", "diploid"]}
print("data:", data_dir, counts)
print("code:", code_dir)
assert counts == {"haploid": 1230, "diploid": 1770}, "unexpected number of images"

# /kaggle/input is read-only, so the code runs from /kaggle/working, which is saved as output.
PROJECT = Path("/kaggle/working/project")
PROJECT.mkdir(parents=True, exist_ok=True)
for f in code_dir.glob("*.py"):
    shutil.copy(f, PROJECT)
os.environ["HAPLOID_DATA_DIR"] = str(data_dir)
os.environ["PYTHONWARNINGS"] = "ignore"
RESULTS = PROJECT / "results"
LOG = RESULTS / "train_log.txt"


def run(*args):
    """Run a project script, show its output live and append it to results/train_log.txt."""
    RESULTS.mkdir(exist_ok=True)
    print("$ python", " ".join(args), flush=True)
    with subprocess.Popen([sys.executable, "-u", *args], cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as p, open(LOG, "a") as log:
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if p.returncode:
        raise RuntimeError(f"{args[0]} failed with exit code {p.returncode}")

In [ ]:
# Same stratified split as the original run: the test set must have 184 haploid and 266 diploid images.
run("01_make_splits.py")
import pandas as pd

splits = pd.read_csv(RESULTS / "splits.csv")
test_counts = splits.loc[splits["split"] == "test", "class"].value_counts().to_dict()
print("test set:", test_counts)
assert test_counts == {"diploid": 266, "haploid": 184}, "the split differs from the original run; stop here."

In [ ]:
# Quick checks: the quantum circuit test, then a tiny 1-epoch training run.
run("check_quantum_circuit.py")
run("04_finetune.py", "--head", "qcnn", "--seed", "0", "--epochs", "1", "--smoke")

In [ ]:
# Full training: 2 heads x 3 seeds, 12 epochs each. Each run prints its test result at the end.
t0 = time.time()
for head in ["qcnn", "classical"]:
    for seed in [0, 1, 2]:
        run("04_finetune.py", "--head", head, "--seed", str(seed), "--epochs", "12")
        print(f"--- {head} seed {seed} done, {(time.time() - t0) / 60:.0f} min so far ---\n", flush=True)

In [ ]:
# Plain and TTA predictions, then the final table (all choices are made on validation only).
run("05_evaluate.py")
run("06_report.py")

In [ ]:
# Package the results. The trained models stay in project/results/finetune/*/best_model.pt;
# the image cache and resume checkpoints are only needed during training, so they are deleted.
(RESULTS / "environment.txt").write_text(
    f"GPU: {gpu}\nPython: {sys.version}\n\n"
    + subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
for f in [RESULTS / "image_cache_224.pt", *RESULTS.rglob("checkpoint.pt")]:
    f.unlink(missing_ok=True)
shutil.rmtree(RESULTS / "finetune_smoke", ignore_errors=True)

with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file() and f.suffix != ".pt":
            z.write(f, f.relative_to(PROJECT))
print("saved /kaggle/working/results.zip")
print(pd.read_csv(RESULTS / "final_table.csv").round(4).to_string(index=False))